<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">دو بلوک مستقل یا یک بلوک دوبار؟</h1>
<p style="text-align:right">درس 47 از 92 · چرا چند بلوک پشت سر هم می‌گذاریم؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">41-stack</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-04/41-stack.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right"><bdi dir="ltr">Stack</bdi> ثبت‌شده با وزن‌های مستقل بسازید و تعداد <bdi dir="ltr">Parameter</bdi>هایش را کنترل کنید.</p><p style="text-align:right"><span class="phrase-lead" style="white-space:nowrap">پیش‌نیاز: ساخت</span> نمونهٔ کلاس، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">ModuleList</code> و قرارداد ورودی/خروجی بلوک را بشناسید.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۶۰–۱۰۵ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">nn.ModuleList([block,block])</code> چند مجموعه وزن مستقل دارد؟ اگر یک بلوک را تغییر دهید، بلوک دوم چه می‌شود؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from torch import nn
from mini_gpt.config import ModelConfig
from mini_gpt.transformer import TransformerBlock
config = ModelConfig(12,8,8,2,1,0.)
prototype = TransformerBlock(config)
per_block = sum(p.numel() for p in prototype.parameters())
print('one block parameters:',per_block)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">make_stack(config, count)</code> یک <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">nn.ModuleList</code> با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">count</code> نمونهٔ مستقل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">TransformerBlock</code> بسازد. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">count</code> در این تمرین مثبت است؛ از تکرار یک شیء در فهرست استفاده نکنید.</p>
</div>

In [ ]:
def make_stack(config, count):
    # TODO
    return None

In [ ]:
def test_exercise():
    stack = make_stack(config,2)
    if stack is None: return False
    assert isinstance(stack,nn.ModuleList) and len(stack) == 2
    assert stack[0] is not stack[1]
    assert stack[0].attention.qkv.weight is not stack[1].attention.qkv.weight
    container = nn.Module(); container.blocks = stack
    assert sum(p.numel() for p in container.parameters()) == 2*per_block
    before = stack[1].attention.qkv.weight.detach().clone()
    with torch.no_grad(): stack[0].attention.qkv.weight.add_(1.)
    assert torch.equal(stack[1].attention.qkv.weight,before)
    assert sum(p.numel() for p in make_stack(config,3).parameters()) == 3*per_block
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">num_layers</code> را در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">MiniGPT</code> از ۱ به ۲ و ۳ تغییر دهید. جدول‌های ورودی و خروجی فقط یک بار ساخته می‌شوند؛ انتظار برابری <bdi dir="ltr">Logits</bdi> مدل‌های تازه نداریم.</p>
</div>

In [ ]:
from mini_gpt.model import MiniGPT
for L in (1,2,3):
    model = MiniGPT(ModelConfig(12,8,8,2,L,0.)).eval()
    print('layers, parameters:',L,sum(p.numel() for p in model.parameters()))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">نسخهٔ خراب یک نمونه را دوبار ثبت می‌کند. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">independent_pair(config)</code> دو بلوک مستقل در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">ModuleList</code> برگرداند؛ صرفاً کپی‌کردن فهرست <bdi dir="ltr">Python</bdi> کافی نیست.</p>
</div>

In [ ]:
shared = TransformerBlock(config)
wrong = nn.ModuleList([shared,shared])
print('same object:',wrong[0] is wrong[1])
print('unique parameters:',sum(p.numel() for p in wrong.parameters()),'expected independent:',2*per_block)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def independent_pair(config):
    # TODO
    return None

In [ ]:
def test_repair():
    result = independent_pair(config)
    if result is None: return False
    assert isinstance(result,nn.ModuleList) and len(result) == 2
    assert result[0] is not result[1]
    assert sum(p.numel() for p in result.parameters()) == 2*per_block
    assert not ({id(p) for p in result[0].parameters()} & {id(p) for p in result[1].parameters()})
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">MiniGPT.blocks</code> از همین الگوی ساخت مستقل استفاده می‌کند. <bdi dir="ltr">v6</bdi> تنظیم کوچکِ مدل نهایی است؛ <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">ModuleList</code> <bdi dir="ltr">Parameter</bdi>ها را ثبت می‌کند، اما حلقهٔ <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">forward</code> باید جداگانه خروجی هر بلوک را به بعدی بدهد.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">چرا تعداد فراخوانی بلوک لزوماً تعداد مجموعه‌های مستقل <bdi dir="ltr">Parameter</bdi> را نشان نمی‌دهد؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-04/41-stack.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/41-stack.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>